In [1]:
import pandas as pd
import numpy as np
import csv
import os

In [3]:
dfOrigin = pd.read_csv('BID_final.csv', encoding='cp949')
dfMethod = pd.read_csv('workbookdata/공고결과_낙찰방법/Total_공고결과_낙찰방법.csv', encoding='cp949')

In [4]:
dfOrigin.head(1)

,bidID,참가수,공고번호,공고차수,판단번호,항목번호,입찰명,발주기관코드,발주기관,개찰일시,낙찰업체코드,낙찰업체상호,대표자,업체주소,낙찰률,낙찰금액,카테고리,pick_keywords,군 분류
0,1,81,SFL0160,1,47669,***,임시명찰 등 2품목 구매,SFL,제3283부대,2016-12-30 12:30:00,H84F8,주식회사 전우밀리터리,이정구,"경기도 고양시 덕양구 통일로140, 1층 에이143호, 2층 에이261호,지하1층 ...",88.0062,23689950.0,2종,명찰,육군


In [5]:
dfMethod.head(1)

,공고구분,공고번호,공고차수,판단번호,항목번호,입찰명,요구년도,발주기관코드,발주기관,계약방법,입찰방법,낙찰자결정방법,개찰일시,개찰일자,입찰결과
0,정상공고,PIP0001,1,9100,***,중요시설경계시스템 1차사업,2016,EGH,지휘정찰계약팀 (구:19.09.16),협상에의한계약(서류),총액제,최저가격제,2016-12-30 12:00:00,2016-12-30,낙찰


In [36]:
# 병합 수행 (left join)
dfMethod_unique = dfMethod.drop_duplicates(subset=['공고번호', '판단번호'], keep='first')
merged_df = pd.merge(dfOrigin, dfMethod_unique[['공고번호', '판단번호', '항목번호', '계약방법']], 
                     on=['공고번호', '항목번호', '판단번호'], how='left')

# NaN 값을 '확인안됨'으로 채움
merged_df['계약방법'] = merged_df['계약방법'].fillna('확인안됨')

# 결과 확인 (merged_df가 업데이트된 dfOrigin 역할을 함)
merged_df

,bidID,참가수,공고번호,공고차수,판단번호,항목번호,입찰명,발주기관코드,발주기관,개찰일시,낙찰업체코드,낙찰업체상호,대표자,업체주소,낙찰률,낙찰금액,카테고리,pick_keywords,군 분류,계약방법
0,1,81,SFL0160,1,47669,***,임시명찰 등 2품목 구매,SFL,제3283부대,2016-12-30 12:30:00,H84F8,주식회사 전우밀리터리,이정구,"경기도 고양시 덕양구 통일로140, 1층 에이143호, 2층 에이261호,지하1층 ...",88.0062,2.368995e+07,2종,명찰,육군,일반경쟁
1,2,1,LFC0043,1,45002,1,장거리 이동 신병 도시락 제조납품,LFC,육군훈련소,2016-12-30 10:00:00,GA6E7,주식회사 엘엔에프 농업회사법인,이용환,충청북도 옥천군 옥천읍 삼청리877-1,96.8333,5.810000e+03,1종,도시락,육군,확인안됨
2,3,22,LGT0034,1,48136,***,0부대 중대시설 분전반 제조설치(16-312),LGT,제2307부대,2016-12-29 11:00:00,F91A4,설악테크,김화성,강원특별자치도 인제군 북면 원통로74번길10-5(원통농공단지),88.0408,2.909872e+07,4종,제조설치,육군,제한경쟁
3,4,22,UMM0839,1,48049,***,13-본-국대-01 00학교 가구 및 비품 제조설치,UMM,국군재정관리단,2016-12-29 11:00:00,B30CC,주식회사 디에스나이키,김준홍,"서울특별시 구로구 구로중앙로134, 4층(구로동, 리치몰)",88.2297,2.165042e+09,2종,가구,기타,제한경쟁
4,5,9,LGT0035,1,48138,***,00부대 중대시설 조명기구 제조납품(16-314),LGT,제2307부대,2016-12-29 11:00:00,C959B,주식회사 한서,이명옥,강원특별자치도 원주시 태장공단길47-0 (태장동),88.3713,3.242441e+07,4종,조명기구,육군,제한경쟁
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74725,74728,164,SCQ0154,2,54928,***,수영장자동청소기 등 16종 구매,SCQ,제9911부대,2024-01-03 10:30:00,EF2CA,대광종합상사,김정현,강원특별자치도 강릉시 경강로2496,89.4840,2.548920e+07,2종,청소기,해병,확인안됨
74726,74729,33,HCL0005,1,51,***,24년 간부식당 식자재 구매,HCL,공군사관학교,2024-01-03 10:30:00,D3B5E,미가온,변정순,경기도 파주시 장명산길174-20 (오도동),83.3270,3.519749e+08,1종,부식,공군,확인안됨
74727,74730,3,SCR0090,2,61362,***,24년 배추김치(포기절단) 등 10종 구매,SCR,해군제3함대사령부,2024-01-03 10:30:00,J47CB,현진식품 영농조합법인,정재웅,전라남도 무안군 망운면 조금나루길22,88.9890,5.389600e+07,1종,김치,해군,제한경쟁
74728,74731,31,MDS0030,1,66867,***,교육단 증개축 GHP 실외기 설치납품,MDS,제3707부대,2024-01-03 10:30:00,E1986,(주)비케이에너지,이복경,경기도 의정부시 민락로387-0 (낙양동),88.0390,6.049600e+07,2종,교육,육군,제한경쟁


In [37]:
merged_df.계약방법.value_counts()

계약방법
확인안됨           46603
제한경쟁           19994
일반경쟁            7256
2단계경쟁(동시)        486
지명경쟁             236
협상에의한계약(전자)      122
협상에의한계약(서류)       21
2단계경쟁(분리)          7
수의계약               5
Name: count, dtype: int64

In [11]:
len(dfOrigin)

74730

In [12]:
len(merged_df)

90468

In [38]:
merged_df.to_csv('BID_Method_Raw.csv', index=False, encoding='cp949')

In [39]:
df = pd.read_csv('BID_Method_Raw.csv', encoding='cp949')
result_df = df[~df['계약방법'].str.contains('확인안됨', na=False, regex=True)]


In [34]:
result_df.계약방법.value_counts()

계약방법
제한경쟁           33050
일반경쟁            8240
2단계경쟁(동시)        619
지명경쟁             232
수의계약             147
협상에의한계약(전자)      109
협상에의한계약(서류)       14
2단계경쟁(분리)          7
Name: count, dtype: int64

In [40]:
result_df.to_csv('BID_Method_unknownRemoved.csv', index=False, encoding='cp949')